# 07 · External tests (recomputed with the final model)

| Test | Boreholes | Cells |
|---|---|---|
| **A. V3 holdout** | 11 Downtown boreholes kept out before any modelling | V3 rule |
| **B. New boreholes** | the 10 boreholes received after V3 was frozen | V3 rule (only ST samples have a USCS symbol, so the SS gaps stay empty) |
| **C. New boreholes, SS gaps filled** | the same 10 | the student's rule: an unclassified SS sample between two samples of the same class takes that class (decided after B was seen; a sensitivity result) |

The **final model** and the **comparison model** (notebook 05) are both scored, so the thesis can show them side by side. V3 scored these boreholes once with GWXGB; recomputing GWXGB gives the same numbers (notebook 10 checks this). If the final model is not V3's model (an override), its external scores were obtained **after** V3's results were known: report that.

In a quick test run the external boreholes are never scored: development boreholes stand in for them (as in V3's smoke run).

**Run time:** about 1 minute.

In [ ]:
import json, sys, time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Image

sys.path.insert(0, str(next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "settings.toml").exists())))
import helpers as H

S = H.load_settings()
T0 = time.time()
print("QUICK TEST RUN - not thesis results" if S["run"]["quick"] else "FULL RUN", "| V3 folder:", S["v3_root"])
import joblib

cells = pd.read_csv(H.processed_dir(S) / "cells.csv")
dev = cells[cells.set == "development"]
rec = json.loads((H.out_dir(S, "05_final_model") / "final_model.json").read_text())
model_path = H.out_dir(S, "05_final_model", private=True) / rec["model_file"]
assert H.sha256(model_path) == rec["model_sha256"], "the final model file has changed"
model, thr = joblib.load(model_path), rec["threshold"]
cmp_rec = json.loads((H.out_dir(S, "05_final_model") / "comparison_model.json").read_text())
cmp_model = H.load_v3_model(S, H.out_dir(S, "05_final_model", private=True) / "comparison_model.joblib")   # may be V3's frozen model
MODELS = [(rec["model"], model, thr), (cmp_rec["model"], cmp_model, cmp_rec["threshold"])]
dev_xy = dev.drop_duplicates("BH_Code")[["UTM_Easting_m", "UTM_Northing_m"]].values
OUT, PRIV = H.out_dir(S, "07_external_tests"), H.out_dir(S, "07_external_tests", private=True)
dev_codes = dev.BH_Code.drop_duplicates().sort_values().values
print("Final model:", rec["model"], "| threshold", thr, "| comparison:", cmp_rec["model"], "| threshold", cmp_rec["threshold"])

def score_test(test, name):
    """Scores of both models on all cells and on measured cells only; overall and by group."""
    preds, rows = [], []
    for mname, m, th in MODELS:
        p = m.predict_proba(test[H.FEATURES].values)[:, 1]
        t = test.assign(p_clay=p, y_pred=(p >= th).astype(int), model=mname)
        preds.append(t)
        for subset, d0 in (("all cells", t), ("measured only", t[t.measured == 1])):
            rows.append({"test": name, "model": mname, "cells": subset, "group": "all", **H.scores(d0.y, d0.p_clay, d0.y_pred)})
            for g, d in d0.groupby("group"):
                rows.append({"test": name, "model": mname, "cells": subset, "group": g, **H.scores(d.y, d.p_clay, d.y_pred)})
    return pd.concat(preds, ignore_index=True), pd.DataFrame(rows)

## A. V3 external holdout (11 boreholes)

In [ ]:
holdout = H.load_v3_holdout(S)
if S["run"]["quick"]:
    stand_in = np.random.default_rng(0).choice(dev_codes, 8, replace=False)
    test_a = cells[cells.BH_Code.isin(stand_in)].assign(group="smoke_stand_in")
else:
    test_a = cells[cells.set == "external"].assign(group=lambda d: d.BH_Code.map(holdout.group))
test_a = test_a.assign(dist_nearest_train_m=H.nearest_distance(test_a[["UTM_Easting_m", "UTM_Northing_m"]].values, dev_xy))
pred_a, res_a = score_test(test_a, "A. V3 holdout")

## B and C. The 10 new boreholes

In [ ]:
BANDS, LABELS = [0, 300, 700, 1000, np.inf], ["< 300 m", "300-700 m", "700-1000 m", "> 1000 m"]
new_records, info = H.load_new_external(S)
print(info)
results, preds_new = [res_a], [pred_a.assign(test="A")]
for name, blocks, seed in (("B. new boreholes", True, 1), ("C. new boreholes, SS gaps filled", False, 1)):
    S2 = {**S, "cells": {**S["cells"], "unclassified_blocks_fill": blocks}}
    c, _ = H.make_cells(new_records, S2)
    c, fi = H.fill_gaps(c, new_records, S2)
    if S["run"]["quick"]:
        stand_in = np.random.default_rng(seed).choice(dev_codes, 8, replace=False)
        test = cells[cells.BH_Code.isin(stand_in)].copy()
    else:
        test = c
    print(f"{name}: {test.BH_Code.nunique()} boreholes, {len(test)} cells ({int(test.measured.sum())} measured)")
    test = test.assign(dist_nearest_train_m=H.nearest_distance(test[["UTM_Easting_m", "UTM_Northing_m"]].values, dev_xy))
    test["group"] = pd.cut(test.dist_nearest_train_m, BANDS, right=False, labels=LABELS).astype(str)
    p, r = score_test(test, name)
    results.append(r); preds_new.append(p.assign(test=name[0]))
res = pd.concat(results, ignore_index=True)
res.to_csv(OUT / "external_tests.csv", index=False)
pd.concat(preds_new, ignore_index=True).to_csv(PRIV / "external_predictions.csv", index=False)
cols = ["test", "model", "cells", "n", "accuracy", "balanced_accuracy", "macro_f1", "recall_clay", "recall_sand", "roc_auc", "brier"]
display(res[res.group == "all"][cols].round(3))
display(res[res.group != "all"][cols + ["group"]].round(3))

## Per borehole (for Fig. 9b)

In [ ]:
allp = pd.concat(preds_new, ignore_index=True)
per = (allp.assign(correct=(allp.y == allp.y_pred).astype(float))
           .groupby(["test", "model", "BH_Code"])
           .agg(cells=("y", "size"), clay_share=("y", "mean"), accuracy=("correct", "mean"),
                dist_nearest_train_m=("dist_nearest_train_m", "first")).reset_index())
per.to_csv(PRIV / "external_by_borehole.csv", index=False)
display(per.pivot_table(index=["test", "BH_Code", "dist_nearest_train_m"], columns="model", values="accuracy").round(3))
print(f"Finished in {time.time() - T0:.0f} s. Next: notebook 08.")